## Цель работы

Реализовать систему, которая:

- распознаёт речевые команды на русском языке;
- использует минимум 5 команд;
- выбирает текстовый ответ из заранее подготовленного словаря;
- измеряет точность распознавания команд;
- измеряет время обработки и проверяет требование задержки менее 5 секунд.

В качестве ASR используется **Vosk** с русскоязычной моделью. Такой вариант соответствует материалам курса: Vosk рассматривается как офлайн-система распознавания речи с невысокими системными требованиями и возможностью быстрой обработки.

**Обратите внимание:** порог >90% нельзя честно объявить выполненным без реального набора голосовых записей.

В работе предусмотрен автоматический тест на размеченных WAV-файлах, который выдаёт фактическую accuracy.


### Связь с материалами урока

В уроках показаны:

- преобразование аудио в WAV с параметрами 16 кГц, mono, PCM;
- распознавание через `KaldiRecognizer`;
- офлайн-модель Vosk;
- оценка качества ASR через WER/CER и оценка задержки;
- использование языкового контекста для уменьшения ошибок.

В этой работе добавляется прикладной слой **распознавание команды -> выбор ответа из словаря**.

In [ ]:
# Установка библиотек

!pip install -q vosk pydub jiwer
!apt-get -qq update
!apt-get -qq install -y ffmpeg


In [ ]:
# Импорты и настройки

import base64
import difflib
import json
import random
import subprocess
import time
import wave
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import Audio, display
from google.colab import files, output
from vosk import Model, KaldiRecognizer, SetLogLevel

SEED = 42
random.seed(SEED)

MODEL_URL = "https://alphacephei.com/vosk/models/vosk-model-small-ru-0.22.zip"
MODEL_DIR = Path("/content/model")
MODEL_ZIP = Path("/content/vosk-model-small-ru-0.22.zip")

SAMPLE_RATE = 16000
MAX_RESPONSE_TIME = 5.0

SetLogLevel(-1)

print("Конфигурация готова.")


In [ ]:
# Словарь команд и заранее подготовленных ответов

COMMANDS = {
    "привет": {
        "phrases": ["привет", "здравствуй", "добрый день", "добрый вечер"],
        "responses": [
            "Здравствуйте! Рада вас слышать.",
            "Привет! Чем могу помочь?",
            "Здравствуйте! Я готова выполнить команду."
        ],
    },
    "время": {
        "phrases": ["который час", "скажи время", "сколько сейчас времени"],
        "responses": [
            "Команда «время» распознана. В этой версии ответ выбирается из подготовленного словаря.",
            "Сейчас активна команда определения времени. Для демонстрации используется статический ответ."
        ],
    },
    "погода": {
        "phrases": ["какая погода", "расскажи про погоду", "что с погодой"],
        "responses": [
            "Команда «погода» распознана. В этой версии ответ берётся из заранее подготовленного словаря.",
            "Погода: для демонстрации используется статический ответ."
        ],
    },
    "сказка": {
        "phrases": ["расскажи сказку", "расскажи мне сказку", "хочу сказку"],
        "responses": [
            "Жили-были ёжик и заяц. Однажды они отправились искать волшебный лес и нашли там источник, исполняющий добрые желания.",
            "В одном маленьком лесу жил любопытный лисёнок. Однажды он нашёл старую карту и отправился навстречу приключениям.",
            "Далеко-далеко жил маленький дракон, который мечтал научиться дружить с людьми."
        ],
    },
    "анекдот": {
        "phrases": ["расскажи анекдот", "расскажи шутку", "хочу анекдот"],
        "responses": [
            "Программист пришёл в библиотеку и спросил: «Есть книги про Java?» Библиотекарь ответила: «Конечно. А вам кофе или чай?»",
            "Почему программист любит тёмную тему? Потому что светлая тема привлекает слишком много багов."
        ],
    },
    "помощь": {
        "phrases": ["помощь", "что ты умеешь", "какие команды ты знаешь"],
        "responses": [
            "Я понимаю команды: привет, время, погода, сказка, анекдот, помощь и стоп.",
            "Доступны голосовые команды для приветствия, времени, погоды, сказки, анекдота, помощи и остановки."
        ],
    },
    "стоп": {
        "phrases": ["стоп", "остановись", "завершить работу"],
        "responses": [
            "Работа голосового помощника остановлена.",
            "Хорошо, завершаю работу."
        ],
    },
}

PHRASE_TO_COMMAND = {
    phrase: command
    for command, data in COMMANDS.items()
    for phrase in data["phrases"]
}

GRAMMAR_PHRASES = list(PHRASE_TO_COMMAND.keys())

print(f"Команд: {len(COMMANDS)}")
for command, data in COMMANDS.items():
    print(f"- {command}: {len(data['phrases'])} вариантов")


In [ ]:
# Загрузка русской модели Vosk

if not MODEL_DIR.exists():
    if not MODEL_ZIP.exists():
        print("Скачиваем русскую модель Vosk...")
        subprocess.run(
            ["wget", "-q", MODEL_URL, "-O", str(MODEL_ZIP)],
            check=True
        )

    print("Распаковываем модель...")
    with zipfile.ZipFile(MODEL_ZIP, "r") as zf:
        zf.extractall("/content")

    extracted = Path("/content/vosk-model-small-ru-0.22")
    if extracted.exists():
        extracted.rename(MODEL_DIR)

print("Модель:", MODEL_DIR)


In [ ]:
# Загрузка модели один раз

model = Model(str(MODEL_DIR))
print("Модель Vosk загружена.")


In [ ]:
# Подготовка аудио

def convert_to_wav(input_path, output_path="/content/command.wav"):
    """Приводит аудио к формату 16 кГц, mono, PCM S16LE."""
    command = [
        "ffmpeg",
        "-y",
        "-i", str(input_path),
        "-ar", str(SAMPLE_RATE),
        "-ac", "1",
        "-c:a", "pcm_s16le",
        str(output_path),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.decode("utf-8", errors="ignore"))

    return Path(output_path)


def normalize_text(text):
    text = text.lower().strip()
    return " ".join(text.split())


In [ ]:
# Распознавание речи с ограниченным словарём команд

def recognize_command(audio_path):
    """
    Распознаёт WAV и выбирает команду из заранее заданного набора.
    Ограниченная грамматика уменьшает пространство поиска.
    """
    with wave.open(str(audio_path), "rb") as wf:
        sample_rate = wf.getframerate()
        channels = wf.getnchannels()

        if channels != 1:
            raise ValueError("WAV должен быть mono. Используйте convert_to_wav().")

        grammar = json.dumps(GRAMMAR_PHRASES, ensure_ascii=False)
        recognizer = KaldiRecognizer(model, sample_rate, grammar)
        recognizer.SetWords(True)

        parts = []

        while True:
            data = wf.readframes(4000)

            if not data:
                break

            if recognizer.AcceptWaveform(data):
                result = json.loads(recognizer.Result())
                if result.get("text"):
                    parts.append(result["text"])

        final_result = json.loads(recognizer.FinalResult())

        if final_result.get("text"):
            parts.append(final_result["text"])

    text = normalize_text(" ".join(parts))

    if not text:
        return "", None

    if text in PHRASE_TO_COMMAND:
        return text, PHRASE_TO_COMMAND[text]

    candidates = difflib.get_close_matches(
        text,
        GRAMMAR_PHRASES,
        n=1,
        cutoff=0.55,
    )

    if candidates:
        phrase = candidates[0]
        return text, PHRASE_TO_COMMAND[phrase]

    return text, None


In [ ]:
# Выбор ответа из словаря

def get_response(command):
    if command not in COMMANDS:
        return "Команда не распознана. Скажите «помощь», чтобы увидеть список команд."

    return random.choice(COMMANDS[command]["responses"])


def process_audio(audio_path):
    start = time.perf_counter()

    recognized_text, command = recognize_command(audio_path)
    response = get_response(command)

    elapsed = time.perf_counter() - start

    return {
        "recognized_text": recognized_text,
        "command": command,
        "response": response,
        "latency_sec": elapsed,
        "under_5_sec": elapsed < MAX_RESPONSE_TIME,
    }


## Демонстрация на аудиофайле

Нужно загрузить короткую запись команды на русском языке. Лучше использовать WAV/MP3 длительностью примерно 1–3 секунды и произнести одну команду без длинной паузы.

In [ ]:
# Загрузка аудиофайла и демонстрация

uploaded = files.upload()

if uploaded:
    input_name = next(iter(uploaded))
    wav_path = convert_to_wav(input_name, "/content/demo_command.wav")

    result = process_audio(wav_path)

    print("Распознано:", result["recognized_text"])
    print("Команда:", result["command"])
    print("Ответ:", result["response"])
    print(f"Задержка: {result['latency_sec']:.3f} сек.")
    print(
        "Требование < 5 сек.:",
        "ВЫПОЛНЕНО" if result["under_5_sec"] else "НЕ ВЫПОЛНЕНО"
    )

    display(Audio(filename=str(wav_path)))


## Запись команды через микрофон

Запись выполняется прямо в коллабе. После запуска ячейки браузер попросит доступ к микрофону. По умолчанию записываются 4 секунды.

In [ ]:
# Запись команды с микрофона в Google Colab

def record_microphone(seconds=4):
    js = f"""
    (async () => {{
        const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = event => chunks.push(event.data);
        recorder.start();

        await new Promise(resolve => setTimeout(resolve, {seconds * 1000}));

        recorder.stop();
        await new Promise(resolve => {{
            recorder.onstop = resolve;
        }});

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {{type: 'audio/webm'}});
        const buffer = await blob.arrayBuffer();
        const bytes = new Uint8Array(buffer);

        let binary = '';
        const chunkSize = 0x8000;

        for (let i = 0; i < bytes.length; i += chunkSize) {{
            binary += String.fromCharCode(
                ...bytes.subarray(i, i + chunkSize)
            );
        }}

        return btoa(binary);
    }})()
    """

    data = output.eval_js(js)

    audio_bytes = base64.b64decode(data)

    webm_path = "/content/microphone.webm"

    with open(webm_path, "wb") as f:
        f.write(audio_bytes)

    return convert_to_wav(webm_path, "/content/microphone.wav")


# Пример запуска:
# mic_wav = record_microphone(seconds=4)
# result = process_audio(mic_wav)
# print(result)


## Подготовка тестового набора

Для объективной оценки нужно создать каталог:

```text
/content/test_audio/
    привет/
        01.wav
        02.wav
    время/
        01.wav
        02.wav
    погода/
        01.wav
        02.wav
    сказка/
        01.wav
        02.wav
    анекдот/
        01.wav
        02.wav
    помощь/
        01.wav
        02.wav
    стоп/
        01.wav
        02.wav
```

Каждый WAV должен содержать одну произнесённую команду. Для проверки порога 90% желательно иметь не менее 10 записей на каждую команду и несколько вариантов произношения.

Можно подготовить ZIP с этой структурой и загрузить его следующей ячейкой.

In [ ]:
# Загрузка ZIP с тестовыми аудиозаписями

uploaded_test = files.upload()

if uploaded_test:
    zip_name = next(iter(uploaded_test))

    if zip_name.lower().endswith(".zip"):
        test_root = Path("/content/test_audio")
        test_root.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as zf:
            zf.extractall(test_root)

        print("Тестовый набор распакован в:", test_root)
    else:
        print("Для этой ячейки нужен ZIP-файл.")


## Оценка точности распознавания

Используем **accuracy по командам**:

`accuracy = число правильно распознанных команд / общее число записей`.

Это именно прикладная точность классификации команд. Для общего качества ASR в материалах курса также рассматриваются WER и CER.

In [ ]:
# Автоматическая оценка accuracy на размеченных WAV

TEST_DIR = Path("/content/test_audio")


def evaluate_test_set(test_dir=TEST_DIR):
    rows = []

    if not test_dir.exists():
        print(f"Каталог {test_dir} не найден.")
        print("Добавьте тестовые WAV-файлы и повторите запуск.")
        return pd.DataFrame()

    for expected_command_dir in sorted(test_dir.iterdir()):
        if not expected_command_dir.is_dir():
            continue

        expected_command = expected_command_dir.name

        for audio_path in sorted(expected_command_dir.glob("*.wav")):
            start = time.perf_counter()

            recognized_text, predicted_command = recognize_command(audio_path)

            latency = time.perf_counter() - start

            rows.append({
                "file": audio_path.name,
                "expected": expected_command,
                "recognized_text": recognized_text,
                "predicted": predicted_command,
                "correct": predicted_command == expected_command,
                "latency_sec": latency,
            })

    df_eval = pd.DataFrame(rows)

    if df_eval.empty:
        print("Тестовых WAV-файлов пока нет.")
        return df_eval

    accuracy = df_eval["correct"].mean() * 100
    mean_latency = df_eval["latency_sec"].mean()
    max_latency = df_eval["latency_sec"].max()

    print(f"Количество записей: {len(df_eval)}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Средняя задержка: {mean_latency:.3f} сек.")
    print(f"Максимальная задержка: {max_latency:.3f} сек.")
    print()

    print(
        "Критерий accuracy > 90%:",
        "ВЫПОЛНЕН" if accuracy > 90 else "ПОКА НЕ ВЫПОЛНЕН"
    )

    print(
        "Критерий latency < 5 сек.:",
        "ВЫПОЛНЕН" if max_latency < 5 else "ПОКА НЕ ВЫПОЛНЕН"
    )

    return df_eval


evaluation_df = evaluate_test_set()

if not evaluation_df.empty:
    display(evaluation_df)


In [ ]:
# Accuracy по каждой команде

if not evaluation_df.empty:
    per_command = (
        evaluation_df
        .groupby("expected")["correct"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "accuracy"})
    )

    per_command["accuracy"] *= 100

    display(per_command.round(2))


In [ ]:
# Анализ ошибок распознавания

if not evaluation_df.empty:
    errors = evaluation_df[~evaluation_df["correct"]].copy()

    if errors.empty:
        print("Ошибок на тестовом наборе нет.")
    else:
        print(f"Ошибок: {len(errors)}")
        display(errors)


In [ ]:
# Измерение задержки на нескольких запусках

def benchmark_audio(audio_path, repeats=10):
    values = []

    for _ in range(repeats):
        start = time.perf_counter()
        recognize_command(audio_path)
        values.append(time.perf_counter() - start)

    return {
        "runs": repeats,
        "mean_sec": sum(values) / len(values),
        "min_sec": min(values),
        "max_sec": max(values),
    }


# Пример:
# benchmark_audio("/content/demo_command.wav")


## Итог

В проекте реализованы все требуемые части:

1. **Распознавание речи** — Vosk, русский язык, офлайн.
2. **Не менее 5 команд** — реализовано 7 команд.
3. **Словарь ответов** — для каждой команды подготовлено несколько ответов, один выбирается случайно.
4. **Оценка accuracy** — по реальным размеченным голосовым записям.
5. **Оценка времени отклика** — измеряется время распознавания и обработки команды.
6. **Ограниченная грамматика** — распознавание ограничено допустимыми фразами команд.
7. **Постобработка** — небольшие ошибки текста дополнительно обрабатываются через близость строк.

### Проверка критериев

После добавления тестовых записей в `/content/test_audio` итоговая ячейка должна показать:

- `Accuracy > 90%`;
- `max latency < 5 секунд`.

Если accuracy ниже 90%, нужно улучшить варианты фраз, качество записей или словарь грамматики и повторить измерение. Заявлять выполнение критерия без реального тестового набора некорректно.